In [2]:
from scipy.io import loadmat    # used to turn .mat files into numpy Arrays
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [3]:
def plot_ABP_PPG_signal(ABP, PPG, fs):

    t = np.arange(len(PPG)) / fs    # time axis, in seconds 
    
    plt.figure(figsize=(12,6))

    plt.subplot(2,1,1)
    plt.plot(t, ABP, color='blue')
    plt.title('ABP Signal')
    plt.xlabel('Time [s]')
    plt.ylabel('Normalized ABP ([0,1])')

    plt.subplot(2,1,2)
    plt.plot(t, PPG, color='red')
    plt.title('PPG Signal')
    plt.xlabel('Time [s]')
    plt.ylabel('Light Intensity [V]')

    plt.tight_layout()

    plt.show()

In [4]:
def unwrap_matlab_array(x):
    '''
    Recursively unwrap MATLAB object arrays to a plain Python number
    '''
    x = np.asarray(x)
    while x.dtype == object:
        x = x[0]
    if np.size(x) == 1:
        return x.item()
    return x


In [ ]:
data_path = 'ABP_PPG'
data_file = sorted(os.listdir(data_path))
rows = []
print(len(data_file))
for i in range(1):
    file_path = os.path.join(data_path, data_file[i])
    data = loadmat(file_path)
    print(f'accessing data point {i}')
    for j in range(1):
                
        sp = data['signal_processing'][0,j]
        segment = sp['signal']

        ABP_signal = segment[0]
        PPG_signal = segment[1]

        beats_ppg = sp['struct_features']['ppg'][0,0]
        beats_abp = sp['struct_features']['beats_abp'][0,0]
        beats_masked = np.where(beats_abp == 0, np.nan, beats_abp)  # Replace 0 padding with NaN
        print(np.nanmin(beats_masked[0]))

        print(beats_ppg.dtype.names)
        n_beats = beats_ppg.shape[1]
        
        for beat_idx in range(0,n_beats):
            
            SBP_val = np.nanmax(beats_masked[beat_idx])
            DBP_val = np.nanmin(beats_masked[beat_idx])

            row = {
                'data ID': data_file[i],
                'struct': j,
                'beat_idx': beat_idx,
                'SBP': SBP_val,
                'DBP': DBP_val
            }

            feature_values = []
            beat_features = beats_ppg[0,beat_idx]   # contains all features for the current heartbeat beat_idx

            for num_feat in range(0,len(beats_ppg[0,0])):
                
                feature_location = int(unwrap_matlab_array(beat_features[num_feat]))
                feature_values.append(PPG_signal[feature_location])
                feature_name = beats_ppg.dtype.names[num_feat]
                row[f'{feature_name}'] = PPG_signal[feature_location]

            rows.append(row)
            print(rows)

        df = pd.DataFrame(rows)
        df.to_csv('PPG_features_per_heartbeat.csv')


5857
accessing data point 0
57.49677802919037
('left_margin', 'right_margin', 'sys_peak', 'dia_start', 'dia_peak', 'left', 'right', 'sys_idx', 'dicN_idx', 'skew', 'w', 'y', 'z', 'a', 'b', 'c', 'd', 'e', 'bd', 'scoo', 'sc', 'bcda', 'cw', 'bcdea', 'sdoo', 'cs', 'tc', 'ts', 'td', 'tod', 'tnt', 'ttn', 'auc_sys', 'aac_sys', 'auc_dia', 'aac_dia', 'mean_sbp', 'mean_dbp', 'sbp_class', 'dbp_class')
[{'data ID': '3000714_0035.mat', 'struct': 0, 'beat_idx': 0, 'SBP': np.float64(118.51376695812708), 'DBP': np.float64(57.49677802919037), 'left_margin': np.float64(117.3403633248783), 'right_margin': np.float64(75.29339980013025), 'sys_peak': np.float64(60.82142165672859), 'dia_start': np.float64(74.31556343908959), 'dia_peak': np.float64(74.11999616688146), 'left': np.float64(85.8540324993693), 'right': np.float64(90.74321430457255), 'sys_idx': np.float64(69.81751617830258), 'dicN_idx': np.float64(59.45245075127167), 'skew': np.float64(69.03524708947006), 'w': np.float64(64.53719982868307), 'y': np.